# Landslide Planet Download + PNG Export + HF Upload (Nepal)

Stage 2 of the two-notebook pipeline. Consumes the order log written by **Stage 1** to:

* poll Planet order status,
* download the delivered `analytic_sr_udm2` **clip** raster,
* convert each downloaded `after` clip to a lossless square RGB PNG (band3=R, band2=G, band1=B,
  padded to an exact square with 0-value pixels if needed) with a stretch,
* upload the PNGs of a run as a single zip to Hugging Face (and keep it in the Kaggle working dir).

No folders / no GEE enrichment: every file is written flat with the naming convention
`incident_{incident}_candidate_{candidate}_after.{tif,png}`.

State is fully persisted -> nothing is ever **re-downloaded**:
- **order log** (from stage 1) drives the work queue (only entries with an `order_id`).
- **download log** (`local/download_log.json`) records completions / failures under the
  composite key `f"{incident}_{candidate}"` (the same key style everywhere).

Resumability: re-running this notebook resumes exactly where it left off.

> Do not run simultaneously with Stage 1 (shared remote logs).
> Required secrets: `PLANET_API_KEY`, `HF_TOKEN`.


In [ ]:
import os, io, json, math, re, shutil, glob, zipfile
import threading
import queue
import pandas as pd
import numpy as np
import rasterio
from rasterio.merge import merge as rio_merge
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timedelta, timezone
from requests.adapters import HTTPAdapter, Retry
from kaggle_secrets import UserSecretsClient
from huggingface_hub import HfApi, upload_folder, hf_hub_download

# ---------------- configuration ----------------
HF_REPO      = 'sasudo2/landslides'        # <-- SAME as stage 1
HF_REPO_TYPE = 'dataset'
HF_REVISION  = 'main'
ORDER_LOG_REMOTE    = 'local/order_log.json'      # from stage 1
DOWNLOAD_LOG_REMOTE = 'local/download_log.json'   # written by this notebook

ORDER_STATE_DIR = '/kaggle/working/order_state'   # order log mirror (order_log.json)
STATE_LOCAL     = '/kaggle/working/state'         # download log mirror (download_log.json)

# Flat local dir: each candidate's tif/png is stored flat (no per-candidate subfolders).
# Naming convention is preserved: incident_{inc}_candidate_{cand}_after.tif/.png.
LOCAL_ROOT     = '/kaggle/working/downloads/landslides'   # flat local staging dir
HF_DATA_PREFIX = 'landslide'                              # remote mirror of those files

# PNG export: each downloaded `after` clip is rendered to a lossless RGB PNG (band3=R,
# band2=G, band1=B) with a QGIS-like min-max linear stretch per band, padded to a square
# with 0-value pixels if needed. All PNGs produced in one run are zipped once, saved to
# the Kaggle working dir, and uploaded to HF as a single zip.
PNG_DIR_REMOTE   = 'landslide_pngs'                        # remote folder for the session zips
SESSION_PNG_DIR  = '/kaggle/working/session_pngs'         # staging dir for this run's PNGs
SESSION_ZIP_DIR  = '/kaggle/working'                      # where the session zip is written
STRETCH_LO_PCT, STRETCH_HI_PCT = 2, 98                    # percentile stretch bounds per band (QGIS-style)
PNG_BRIGHTNESS_GAMMA = 0.8                                  # <1 brightens exported PNGs; 1.0 = neutral

LOG_FLUSH_EVERY = 50     # ready completions before re-persisting the download log
MAX_WORKERS    = 8

DATA_API   = 'https://api.planet.com/data/v1'
ORDERS_API = 'https://api.planet.com/compute/ops/orders/v2'
REQUEST_TIMEOUT = 300
SUCCESS_STATES = {'success'}
PENDING_STATES = {'queued', 'running', 'processing', 'pending'}
FAILED_STATES  = {'failed', 'error', 'cancelled'}


In [ ]:
secrets = UserSecretsClient()
PLANET_API_KEY = secrets.get_secret('Lokesh_planet')
HF_TOKEN       = secrets.get_secret('huggingface_token')

def planet_session(api_key):
    s = requests.Session()
    s.auth = (api_key, '')
    r = Retry(total=5, backoff_factor=1.2, status_forcelist=[429, 500, 502, 503, 504],
              allowed_methods=frozenset(['GET', 'POST']), respect_retry_after_header=True)
    s.mount('https://', HTTPAdapter(max_retries=r))
    return s

PLANET = planet_session(PLANET_API_KEY)
HF_API = HfApi(token=HF_TOKEN)
print('sessions ready')


In [ ]:
# STEP 1 - Read order log + download log from HF, build the work queue.
def load_remote_json(remote, default):
    try:
        p = hf_hub_download(repo_id=HF_REPO, repo_type=HF_REPO_TYPE, revision=HF_REVISION,
                            filename=remote, token=HF_TOKEN)
        with open(p) as f:
            return json.load(f)
    except Exception:
        return default

def as_dict(obj):
    if isinstance(obj, dict):
        return obj
    if isinstance(obj, list):
        return {e['key']: e for e in obj if isinstance(e, dict) and 'key' in e}
    return {}

order_log    = as_dict(load_remote_json(ORDER_LOG_REMOTE, {}))
download_log = as_dict(load_remote_json(DOWNLOAD_LOG_REMOTE, {}))
print('order log entries    :', len(order_log))
print('download log entries :', len(download_log))

# Work queue = order-log entries that have an order_id and are NOT already downloaded
# for THIS order_id. If a landslide was re-ordered (new order_id), the stale download
# log entry must NOT block the re-download of the fresh composite imagery.
work = []
for key, en in order_log.items():
    oid = en.get('order_id')
    if not oid:
        continue                       # no order -> nothing to download
    done = download_log.get(key) or {}
    if done.get('status') == 'complete' and done.get('order_id') == oid:
        continue                       # already downloaded for this exact order
    work.append({'key': key, 'entry': en})
print('work queue (to download):', len(work))


In [ ]:
# STEP 2 - Check Planet order status, download + extract delivered clip rasters.
# Stage 1 now packs EVERY covering scene into one order, so an order delivers one
# clipped analytic_sr raster per scene (plus its UDM2 pair). We download ALL of them
# and mosaic into one complete composite; single-scene orders collapse to a no-op.
RA_SUFFIXES   = ('_analyticms_sr_clip.tif', '_sr_clip.tif')
UDM2_SUFFIXES = ('_analyticms_udm2_clip.tif', '_udm2_clip.tif')
ARCHIVE_SUFFIXES = ('.zip', '.tar', '.tar.gz')

def order_status(oid):
    r = PLANET.get(f'{ORDERS_API}/{oid}', timeout=120)
    r.raise_for_status()
    return r.json()

def download_url(url, path):
    tmp = path + '.part'
    with PLANET.get(url, stream=True, timeout=REQUEST_TIMEOUT) as r:
        r.raise_for_status()
        with open(tmp, 'wb') as f:
            for chunk in r.iter_content(chunk_size=1 << 20):
                if chunk:
                    f.write(chunk)
    os.replace(tmp, path)

def get_deliverables(order):
    results = order.get('_links', {}).get('results')
    if not results:
        oid = order.get('id')
        if oid:
            results = order_status(oid).get('_links', {}).get('results', [])
    return results or []

def extract_rasters(order, local_dir):
    # Download every delivered clip raster + UDM2 pair (loose deliverables or inside
    # zip/tar archives) into local_dir. Detection is NAME-AGNOSTIC: any GeoTIFF that is
    # not a UDM2 mask is treated as a scene raster, so it survives Planet's clip-tool
    # naming variants. Returns parallel lists (sr_paths, udm2_paths) paired per scene.
    os.makedirs(local_dir, exist_ok=True)
    dl = [((res.get('name') or '').lower(), res.get('location'))
          for res in get_deliverables(order) if res.get('location')]
    print(f'[deliverables] order returned {len(dl)} files')
    if dl:
        print('  names:', sorted(n for n, _ in dl)[:8])

    direct   = [(n, loc) for n, loc in dl if n.endswith('.tif')]
    archives = [(n, loc) for n, loc in dl if n.endswith(ARCHIVE_SUFFIXES)]

    if direct:
        for n, loc in direct:
            download_url(loc, os.path.join(local_dir, os.path.basename(n)))
    elif archives:
        for i, (n, loc) in enumerate(archives):
            tmp = os.path.join(local_dir, f'_deliver_{i}_{os.path.basename(n)}')
            download_url(loc, tmp)
            if zipfile.is_zipfile(tmp):
                with zipfile.ZipFile(tmp) as z:
                    z.extractall(local_dir)
            else:
                shutil.unpack_archive(tmp, local_dir)
            os.remove(tmp)
    else:
        return [], []

    hits = [f for f in glob.glob(os.path.join(local_dir, '**', '*'), recursive=True)
            if os.path.isfile(f) and f.lower().endswith('.tif')]
    # SR raster = any delivered tif that is NOT a UDM2 mask; UDM2 mask = name contains 'udm'.
    sr  = sorted(f for f in hits if 'udm' not in os.path.basename(f).lower())
    udm = sorted(f for f in hits if 'udm' in os.path.basename(f).lower())
    print(f'[extract] {len(sr)} SR clip(s), {len(udm)} UDM2 clip(s) -> {os.path.basename(local_dir)}')
    return sr, udm

def mask_clear_pixels(sr_path, udm2_path, out_path):
    # Zero out any pixel Planet's UDM2 classifier flags as cloud/shadow/haze/snow
    # (band 1 == 1 means clear) so the composite only carries clear imagery.
    # Mirrors incident_download.ipynb's mask_clear_pixels.
    with rasterio.open(sr_path) as src:
        data = src.read()
        meta = src.meta.copy()
    with rasterio.open(udm2_path) as udm2:
        clear = udm2.read(1)
    if clear.shape != data.shape[1:]:
        print(f'UDM2 mask shape {clear.shape} != SR shape {data.shape[1:]} for {sr_path} - skipping cloud mask')
        shutil.copyfile(sr_path, out_path)
        return
    data[:, clear == 0] = 0
    meta.update(nodata=0)   # mark masked pixels so rio_merge skips them and a covering scene fills in
    with rasterio.open(out_path, 'w', **meta) as dst:
        dst.write(data)

def mosaic_geotiffs(src_paths, dst_path):
    # Merge one or more (already cloud-masked) scene clips into a single raster that
    # covers the full AOI. A single path is just copied through unchanged.
    if len(src_paths) == 1:
        shutil.copyfile(src_paths[0], dst_path)
        return
    srcs = [rasterio.open(path) for path in src_paths]
    try:
        # nodata=0: void / UDM2-masked pixels in one scene are skipped so a clear area
        # from another overlapping scene fills them in, leaving no clipped holes.
        mosaic, out_transform = rio_merge(srcs, nodata=0)
        out_meta = srcs[0].meta.copy()
        out_meta.update({
            'height': mosaic.shape[1],
            'width': mosaic.shape[2],
            'transform': out_transform,
            'count': mosaic.shape[0],
            'nodata': 0,
        })
        with rasterio.open(dst_path, 'w', **out_meta) as dst:
            dst.write(mosaic)
    finally:
        for s in srcs:
            s.close()

def _scene_stem(name):
    # Reduce a delivered clip/Mask filename to a scene-stem that is identical for the
    # SR raster and its UDM2 pair regardless of Planet's naming variant (e.g. both
    # '{scene}_3B_AnalyticMS_SR_clip.tif' and '{scene}_3B_AnalyticMS_udm2_clip.tif'
    # reduce to '{scene}_3b_analyticms').
    import re as _re
    b = os.path.basename(name).lower()
    m = _re.search(r'^(.+?)_(?:3b_)?(?:analyticms_|analytic_)?(?:surface_reflectance_|sr_|sr|udm2_|udm_)?[^/]*clips?\.tif$', b)
    if m:
        return m.group(1)
    for tok in ('_udm2_clip.tif', '_sr_clip.tif', '_clip.tif'):
        if b.endswith(tok):
            return b[:-len(tok)]
    return b.replace('.tif', '')

def build_composite(sr_paths, udm2_paths, dst_path, staging_dir):
    # Mask each scene with its UDM2 pair, then mosaic the clear clips into dst_path.
    # Returns the number of scenes actually merged (>=1) or None on failure.
    os.makedirs(staging_dir, exist_ok=True)
    udm_by_stem = {_scene_stem(p): p for p in udm2_paths}
    masked = []
    for i, sr in enumerate(sr_paths):
        stem = _scene_stem(sr)
        m = os.path.join(staging_dir, f'masked_{i}.tif')
        udm = udm_by_stem.get(stem)
        if udm:
            mask_clear_pixels(sr, udm, m)
        else:
            shutil.copyfile(sr, m)
        masked.append(m)
    try:
        mosaic_geotiffs(masked, dst_path)
    finally:
        for m in masked:
            if os.path.exists(m):
                os.remove(m)
    return len(masked)


In [ ]:
# STEP 2b - Convert a downloaded `after` clip to a lossless square RGB PNG (band3=R,
# band2=G, band1=B) with a QGIS-like min-max linear stretch per band. If the image is
# not square, it is padded to an exact square by adding 0-value pixels. Returns the PNG
# path, or None.
def raster_to_rgb_png(tif_path, png_path, lo_pct=STRETCH_LO_PCT, hi_pct=STRETCH_HI_PCT):
    with rasterio.open(tif_path) as src:
        n = src.count
        # Planet analytic/analytic_sr band order is B,G,R[,N]: choose R,G,B = bands 3,2,1.
        if n >= 3:
            rgb_idx = [1, 2, 3]          # 1-based band numbers -> B is 1, G is 2, R is 3
        else:
            return None
        # Read with the nodata mask so void / UDM2-masked pixels become NaN and are excluded
        # from the stretch (they would otherwise drag vmin down and darken the whole PNG).
        bands = src.read(rgb_idx, masked=True).astype(np.float32).filled(np.nan)
    b, g, r = bands[0], bands[1], bands[2]
    rgb = np.stack([r, g, b], axis=-1)
    out = np.zeros(rgb.shape, dtype=np.uint8)
    for c in range(3):
        ch = rgb[..., c]
        valid = ch[np.isfinite(ch)]
        if valid.size == 0:
            continue
        # QGIS-style MinMax stretch: linear map of the band's actual min/max to 0..255,
        # then apply a gamma < 1 to lift overall brightness.
        vmin = float(valid.min())
        vmax = float(valid.max())
        if vmax <= vmin:
            out[..., c] = 0
            continue
        scale = 255.0 / (vmax - vmin)
        scaled = np.clip((ch - vmin) * scale, 0, 255)
        if PNG_BRIGHTNESS_GAMMA and PNG_BRIGHTNESS_GAMMA != 1.0:
            scaled = 255.0 * (scaled / 255.0) ** PNG_BRIGHTNESS_GAMMA
        out[..., c] = np.clip(scaled, 0, 255).astype(np.uint8)
    # Pad with 0-value pixels to an exact square (all three channels get black padding).
    h, w, _ = out.shape
    if h != w:
        side = max(h, w)
        padded = np.zeros((side, side, 3), dtype=np.uint8)
        padded[:h, :w, :] = out
        out = padded
    # Write as lossless 8-bit RGB PNG (3 bands).
    with rasterio.open(png_path, 'w', driver='PNG', width=out.shape[1], height=out.shape[0],
                       count=3, dtype=np.uint8) as dst:
        dst.write(np.moveaxis(out, -1, 0))
    return png_path


In [ ]:
# STEP 3 - Process one candidate: download raster + render a square RGB PNG. No folders, no GEE.
# Runs in worker threads. Returns (status, key, detail|None).
def process(item):
    en   = item['entry']
    key  = item['key']
    oid  = en['order_id']
    inc, cand = en['incident'], en['candidate']

    try:
        order = order_status(oid)
    except Exception as e:
        return ('poll_error', key, str(e))

    state = (order.get('state') or '').lower()
    if state in PENDING_STATES:
        return ('pending', key, None)        # still compiling -> leave in queue
    if state in FAILED_STATES or state not in SUCCESS_STATES:
        return ('order_failed', key, state)

    # ---- download ALL scene clips, then mosaic into the flat 'after' raster exact-file ----
    # (naming convention preserved). Cleans the per-key staging dir afterwards so the raw
    # scene clips are never bundled into the flat HF upload. Always rebuilds the composite
    # (no exists-guard) so a re-ordered/keyed entry gets the fresh mosaic, never a stale one.
    os.makedirs(LOCAL_ROOT, exist_ok=True)
    after = os.path.join(LOCAL_ROOT, f'incident_{inc}_candidate_{cand}_after.tif')
    staging = os.path.join(LOCAL_ROOT, f'.staging_{inc}_{cand}')
    try:
        sr, udm = extract_rasters(order, staging)
        if not sr:
            return ('no_raster', key, 'no scene clips delivered')
        scene_count = build_composite(sr, udm, after, staging)
        print(f'[composite] {key}: mosaicked {scene_count} scene(s) -> incident_{inc}_candidate_{cand}_after.tif')
    except Exception as e:
        return ('no_raster', key, str(e))
    finally:
        if os.path.exists(staging):
            shutil.rmtree(staging, ignore_errors=True)

    # ---- render the after composite to a square RGB PNG (band3=R, band2=G, band1=B) ----
    try:
        os.makedirs(SESSION_PNG_DIR, exist_ok=True)
        png_path = os.path.join(SESSION_PNG_DIR, f'incident_{inc}_candidate_{cand}_after.png')
        raster_to_rgb_png(after, png_path)
    except Exception as e:
        print(f'PNG render failed for {key}: {e}')

    return ('ready', key, None)


In [ ]:
# STEP 4+5 - Thread-safe driver: workers -> queue; main thread persists the log.
main_q   = queue.Queue()
log_lock = threading.Lock()

def _worker(item):
    try:
        res = process(item)
    except Exception as e:
        res = ('worker_error', item['key'], str(e))
    main_q.put(res)

def now_iso():
    return datetime.now(timezone.utc).isoformat()

def persist_and_upload_log():
    os.makedirs(STATE_LOCAL, exist_ok=True)
    with open(f'{STATE_LOCAL}/download_log.json', 'w') as f:
        json.dump(download_log, f, indent=1, default=str)
    try:
        upload_folder(folder_path=STATE_LOCAL, path_in_repo='local',
                      repo_id=HF_REPO, repo_type=HF_REPO_TYPE, revision=HF_REVISION,
                      commit_message='download log update', token=HF_TOKEN)
        print('[main] download log persisted to HF')
    except Exception as e:
        print('[main] log upload failed (local state kept):', e)

# ---- run: submit workers, drain queue on the main thread, mark done, batch-log ----
# TIFs are uploaded in one final commit after the run (flat layout); only the log is
# flushed periodically (frequent per-file commits on a large flat dir would be wasteful).
total = len(work)
done, logged_since = 0, 0
ready_tifs = []     # local tif paths that completed this run
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
    for it in work:
        ex.submit(_worker, it)
    while done < total:
        res = main_q.get(timeout=None)
        done += 1
        status = res[0]
        if status == 'ready':
            # recompute the flat tif path (status carries no path) - safe from key.
            key = res[1]
            en = order_log[key]
            inc, cand = en['incident'], en['candidate']
            ready_tif = os.path.join(LOCAL_ROOT, f'incident_{inc}_candidate_{cand}_after.tif')
            if os.path.exists(ready_tif):
                ready_tifs.append(ready_tif)
            with log_lock:
                download_log[key] = {'status': 'complete', 'uploaded_at': now_iso(),
                                     'order_id': en.get('order_id'),
                                     'scene_count': en.get('scene_count'),
                                     'aoi_coverage': en.get('aoi_coverage'),
                                     'straddles_scans': en.get('straddles_scans')}
            logged_since += 1
        else:
            dl = {'status': status, 'timestamp': now_iso()}
            if res[2] is not None:
                dl['detail'] = res[2]
            with log_lock:
                download_log[res[1]] = dl
            logged_since += 1
        if logged_since >= LOG_FLUSH_EVERY:
            persist_and_upload_log(); logged_since = 0

# ---- final: upload the flat TIF dir in one commit, then persist the log ----
if ready_tifs:
    try:
        HF_API.upload_folder(
            repo_id=HF_REPO, repo_type=HF_REPO_TYPE, revision=HF_REVISION,
            folder_path=LOCAL_ROOT, path_in_repo=HF_DATA_PREFIX,
            commit_message=f'landslide tif batch ({len(ready_tifs)} files)', token=HF_TOKEN)
        print(f'Uploaded {len(ready_tifs)} new TIFs to HF under {HF_DATA_PREFIX}/')
    except Exception as e:
        print(f'batch TIF upload failed: {e}')   # files kept locally; resume next run

persist_and_upload_log()


In [ ]:
# STEP 6 - Resumability & reporting.
from collections import Counter

skipped_already = sum(1 for k in order_log
                      if (download_log.get(k) or {}).get('status') == 'complete')

status_counts = Counter((dl or {}).get('status') for dl in download_log.values())

print('\n===================== FINAL EXECUTION SUMMARY =====================')
print(f'order-log entries              : {len(order_log)}')
print(f'work queued this run            : {total}')
print(f'skipped (already complete)     : {skipped_already}')

print('\n-- download outcomes (all time, persisted) --')
for s, c in sorted(status_counts.items()):
    print(f'  {s}: {c}')

pending = status_counts.get('pending', 0)
print(f'\npending (still compiling in Planet, will resume next run): {pending}')

print('\n-- failed breakdown (specific reasons) --')
for s in ['order_failed', 'no_raster', 'poll_error', 'batch_tif_upload_failed', 'worker_error']:
    c = status_counts.get(s, 0)
    if c:
        print(f'  {s}: {c}')

print('\n-- fallback_tier_used across successfully ordered candidates --')
tier_break = Counter(order_log[k].get('fallback_tier_used')
                     for k in order_log if order_log[k].get('order_id'))
for t, c in sorted(tier_break.items(), key=lambda x: (x[0] is None, x[0])):
    label = 'none' if t is None else f'tier {t}'
    print(f'  {label}: {c}')

print('\nRe-running this notebook resumes exactly where it left off.')


In [ ]:
# STEP 7 - Zip all PNGs generated in this session, save as a Kaggle output, and upload to HF.
from huggingface_hub import upload_file

def zip_dir_sessions(src_dir, dst_zip):
    n = 0
    with zipfile.ZipFile(dst_zip, 'w', zipfile.ZIP_DEFLATED) as z:
        for f in sorted(glob.glob(os.path.join(src_dir, '*.png'))):
            z.write(f, arcname=os.path.basename(f))
            n += 1
    return n

png_files = glob.glob(os.path.join(SESSION_PNG_DIR, '*.png')) if os.path.exists(SESSION_PNG_DIR) else []
print(f'PNGs generated in this session: {len(png_files)}')

if png_files:
    run_tag = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')
    zip_name = f'landslide_pngs_{run_tag}.zip'
    local_zip = os.path.join(SESSION_ZIP_DIR, zip_name)
    n_archived = zip_dir_sessions(SESSION_PNG_DIR, local_zip)
    print(f'Zipped {n_archived} PNGs -> {local_zip}')
    remote_path = f'{PNG_DIR_REMOTE}/{zip_name}'
    try:
        upload_file(repo_id=HF_REPO, repo_type=HF_REPO_TYPE, revision=HF_REVISION,
                    path_or_fileobj=local_zip, path_in_repo=remote_path, token=HF_TOKEN,
                    commit_message=f'bulk PNG #{run_tag}')
        print(f'Uploaded zip to HF: {remote_path}')
    except Exception as e:
        print(f'Zip upload to HF failed (zip still saved locally): {e}')
else:
    print('No PNGs this session - nothing to zip/upload.')
